In [1]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
import torch
import evaluate

/users/eleves-b/2022/louis.martino/TA_Group-Project/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_feature_names(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]


FEATURES_NO_SCALE = load_feature_names("../data/features/features_no_scale.txt")
FEATURES_SCALE = load_feature_names("../data/features/features_scale.txt")


def prepare_aligned(df):
    """Same rows as text-only training; keeps all columns for structured fusion later."""
    df = df[df["state"].isin(["successful", "failed"])].copy()
    df["text"] = df["name"].fillna("") + " " + df["blurb"].fillna("")
    df["label"] = (df["state"] == "successful").astype(int)
    df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
    return df


train_aligned = prepare_aligned(pd.read_csv("../data/features/train.csv"))
val_aligned = prepare_aligned(pd.read_csv("../data/features/val.csv"))

train_df = train_aligned[["text", "label"]]
val_df = val_aligned[["text", "label"]]

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [3]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 22032/22032 [00:00<00:00, 31262.43 examples/s]


In [5]:
train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")

In [6]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    precision = precision_metric.compute(predictions=predictions, references=labels)
    recall = recall_metric.compute(predictions=predictions, references=labels)

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"],
        "precision": precision["precision"],
        "recall": recall["recall"],
    }

In [7]:
training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipykernel_266409/4030190838.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.519200,0.493600,0.765432,0.811867,0.774267,0.853306
2,0.408400,0.535007,0.767338,0.812523,0.778198,0.850015


TrainOutput(global_step=16524, training_loss=0.46379953318374034, metrics={'train_runtime': 1273.4587, 'train_samples_per_second': 103.805, 'train_steps_per_second': 12.976, 'total_flos': 4377782590783488.0, 'train_loss': 0.46379953318374034, 'epoch': 2.0})

In [9]:
results = trainer.evaluate()
results

{'eval_loss': 0.5350070595741272,
 'eval_accuracy': 0.7673384168482208,
 'eval_f1': 0.8125228586058079,
 'eval_precision': 0.7781981224604175,
 'eval_recall': 0.8500153045607591,
 'eval_runtime': 61.7312,
 'eval_samples_per_second': 356.902,
 'eval_steps_per_second': 44.613,
 'epoch': 2.0}

In [10]:
pred_output = trainer.predict(val_dataset)
preds = np.argmax(pred_output.predictions, axis=-1)

print(preds[:10])

[1 0 1 1 1 1 1 1 1 0]


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score


def build_structured_arrays(aligned_df, scaler=None, *, fit_scaler):
    X_no = aligned_df[FEATURES_NO_SCALE].astype(np.float64).to_numpy()
    X_scale_raw = aligned_df[FEATURES_SCALE].astype(np.float64).to_numpy()
    if fit_scaler:
        scaler = StandardScaler()
        X_scale = scaler.fit_transform(X_scale_raw)
    else:
        assert scaler is not None
        X_scale = scaler.transform(X_scale_raw)
    X = np.hstack([X_no, X_scale])
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X, scaler


X_train_tab, tab_scaler = build_structured_arrays(train_aligned, fit_scaler=True)
X_val_tab, _ = build_structured_arrays(val_aligned, scaler=tab_scaler, fit_scaler=False)


def extract_cls_and_logits(model, tokenizer, texts, batch_size=32, max_length=128):
    device = next(model.parameters()).device
    model.eval()
    cls_chunks, log_chunks = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc, output_hidden_states=True)
            cls = out.hidden_states[-1][:, 0].cpu().numpy()
            logits = out.logits.cpu().numpy()
        cls_chunks.append(cls)
        log_chunks.append(logits)
    return np.vstack(cls_chunks), np.vstack(log_chunks)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

emb_train, logits_train = extract_cls_and_logits(
    model, tokenizer, train_aligned["text"].tolist()
)
emb_val, logits_val = extract_cls_and_logits(
    model, tokenizer, val_aligned["text"].tolist()
)

X_train_fused = np.hstack([emb_train, X_train_tab])
X_val_fused = np.hstack([emb_val, X_val_tab])
y_train = train_aligned["label"].to_numpy()
y_val = val_aligned["label"].to_numpy()

text_val_pred = np.argmax(logits_val, axis=-1)

fusion_clf = LogisticRegression(max_iter=2000, random_state=42)
fusion_clf.fit(X_train_fused, y_train)
fusion_val_pred = fusion_clf.predict(X_val_fused)

text_val_probs = F.softmax(torch.tensor(logits_val), dim=-1)[:, 1].numpy()

fusion_val_probs = fusion_clf.predict_proba(X_val_fused)[:, 1]

print(
    "Text-only DistilBERT (val):",
    {
        "accuracy": float(accuracy_score(y_val, text_val_pred)),
        "f1": float(f1_score(y_val, text_val_pred)),
        "precision": float(precision_score(y_val, text_val_pred)),
        "recall": float(recall_score(y_val, text_val_pred)),
        "auc": float(roc_auc_score(y_val, text_val_probs)),
    },
)

print(
    "Fused [CLS] embedding + structured (val):",
    {
        "accuracy": float(accuracy_score(y_val, fusion_val_pred)),
        "f1": float(f1_score(y_val, fusion_val_pred)),
        "precision": float(precision_score(y_val, fusion_val_pred)),
        "recall": float(recall_score(y_val, fusion_val_pred)),
        "auc": float(roc_auc_score(y_val, fusion_val_probs)),
    },
)
print("Fusion val predictions (first 10):", fusion_val_pred[:10])

Text-only DistilBERT (val): {'accuracy': 0.7673384168482208, 'f1': 0.8125228586058079, 'precision': 0.7781981224604175, 'recall': 0.8500153045607591, 'auc': 0.8390960493676914}
Fused [CLS] embedding + structured (val): {'accuracy': 0.7816811909949165, 'f1': 0.8201734709137132, 'precision': 0.8018274853801169, 'recall': 0.8393786348331803, 'auc': 0.8563565044792987}
Fusion val predictions (first 10): [1 0 0 1 1 1 1 1 1 0]
